# Honger CV: End-to-End Learning and Test Notebook

A guided tour of CV upload and parsing, AI extraction, grounded answers and voice, plus the PayPal and Chapa payment flows. Work through the sections in order. The code cells use local development services and avoid printing credentials.

> **Privacy:** Use a sample or your own CV only if you are comfortable sending it to the configured services. Docling and Ollama can run locally; the CV extraction provider may be a hosted model. Do not save API keys, browser cookies, payment details, or private CV content into this notebook.

This notebook demonstrates the implementation; it does not replace the signed-in product UI for editing a CV or making payments.

## 1. Architecture and prerequisites

The browser CV import flow is implemented in `components/account/cv-import-pipeline.tsx`:

1. Browser uploads the selected file to Next.js `POST /api/cv/parse`.
2. The route tries the configured Resume Parser API first. If it does not return structured CV data, the server sends the file to Docling and normalizes its Markdown.
3. The browser sends that Markdown to `POST /api/cv/extract`; the configured AI provider returns schema-constrained fields.
4. The UI validates and safely merges extracted fields into the draft. Conflicts are shown for review; the owner chooses whether to apply the imported data.
5. Publishing is a separate owner action. The AI knowledge is built from published CV/persona data.

Useful source files: `app/api/cv/parse/route.ts`, `server/services/document-parser/docling-parser.ts`, `app/api/cv/extract/route.ts`, `server/services/cv-extractor/cv-extractor.ts`, and `components/account/cv-import-pipeline.tsx`.

### Start the local services

From the repository root, start PostgreSQL and Docling:

```bash
docker compose up -d postgres docling
```

The local Docling service is exposed on port `5001`. The app defaults to `http://localhost:5001`; if configured, `DOCLING_API_URL` and `DOCLING_API_KEY` are server environment variables.

For the voice parser API and Ollama, first make a local config file if needed:

```bash
cd parser-api
cp .env.example .env
```

Set a private local `API_KEYS` value in `parser-api/.env`, and configure `VOICE_MODELS=openai_compatible/llama3.2`. Then start the API and Ollama and download the model once:

```bash
docker compose up -d api ollama
docker compose exec ollama ollama pull llama3.2
```

The parser container reaches Ollama as `http://ollama:11434/v1`. To run the parser API outside Docker, use `OPENAI_COMPATIBLE_BASE_URL=http://localhost:11434/v1` instead.

Finally, from the repository root, run the web app with its server environment configured (for example, `bun dev`). For a local end-to-end setup, configure `.env.local` with `DOCLING_API_URL=http://localhost:5001`, `RESUME_PARSER_API_URL=http://localhost:8000`, and the same `RESUME_PARSER_API_KEY` as one of the parser API `API_KEYS`. AI extraction separately needs a configured `GEMINI_API_KEY` or `OPENROUTER_API_KEY` for the app's CV extractor.

Never paste production credentials into a notebook. Restart a service after changing its environment.

In [ ]:
from pathlib import Path
from getpass import getpass
import json
import requests

APP_URL = "http://localhost:3000"
DOCLING_URL = "http://localhost:5001"
PARSER_API_URL = "http://localhost:8000"

print("App:", APP_URL)
print("Docling:", DOCLING_URL)
print("Parser API:", PARSER_API_URL)
print("requests version:", requests.__version__)
# If requests is missing in this kernel, install it into the selected kernel:
# %pip install requests

## 2. Check local services

The parser API health route is `/api/v1/health`; Ollama exposes `/api/tags`. Docling-Serve versions can have different health endpoints, so we probe its root and `/docs` rather than assuming a specific health schema. A failed check usually means the container is still starting or the service is not running.

In [ ]:
def probe(label, url):
    try:
        response = requests.get(url, timeout=5)
        print(f"{label}: HTTP {response.status_code} — {url}")
        return response
    except requests.RequestException as exc:
        print(f"{label}: unavailable ({type(exc).__name__}) — {url}")
        return None

app_health = probe("Next.js app", f"{APP_URL}/api/health/")
parser_health = probe("Parser API", f"{PARSER_API_URL}/api/v1/health")
docling_root = probe("Docling", DOCLING_URL)
if not docling_root or not docling_root.ok:
    probe("Docling docs", f"{DOCLING_URL}/docs")
ollama_tags = probe("Ollama", "http://localhost:11434/api/tags")
if ollama_tags and ollama_tags.ok:
    models = ollama_tags.json().get("models", [])
    print("Ollama models:", [model.get("name") for model in models])

## 3. Upload a CV and inspect parsed Markdown

Choose a local PDF, DOCX, PPTX, TXT, or Markdown file (maximum 15 MB). This calls the same Next.js parse route used by the browser. It requires no copied browser cookies; the route enforces same-origin policy for browser writes, while this local notebook request is not a browser request.

The parser route tries the separately configured Resume Parser API first. If that succeeds, its result is converted to Markdown. Otherwise the route calls `DoclingParser`, which sends multipart field `files` to Docling's convert endpoint and normalizes the returned Markdown. The notebook exercises the public app route, so inspect `metadata.extraction_method`/`metadata.parserUsed` to see which configured path handled the file.

In [ ]:
cv_path_text = input("Path to a local CV file (or press Enter to skip): ").strip()
parsed_cv = None
markdown = None

if cv_path_text:
    cv_path = Path(cv_path_text).expanduser()
    if not cv_path.is_file():
        raise FileNotFoundError(f"CV file not found: {cv_path}")
    if cv_path.stat().st_size > 15_000_000:
        raise ValueError("The CV must be smaller than 15 MB.")

    with cv_path.open("rb") as cv_file:
        response = requests.post(
            f"{APP_URL}/api/cv/parse/",
            files={"file": (cv_path.name, cv_file)},
            timeout=150,
        )
    print("HTTP status:", response.status_code)
    if not response.ok:
        print("Error:", response.json().get("error", response.text[:500]))
    else:
        parsed_cv = response.json()
        markdown = parsed_cv.get("markdown", "")
        print("Filename:", parsed_cv.get("filename"))
        print("Metadata:", json.dumps(parsed_cv.get("metadata", {}), indent=2))
        print("Warnings:", parsed_cv.get("warnings", []))
        print("Markdown characters:", len(markdown))
        print("\nMarkdown preview (first 1,500 characters):\n")
        print(markdown[:1500])
else:
    print("Skipped. Set markdown manually in the next section if desired.")

### What the parse response means

- `markdown` is normalized text for downstream extraction and inspection.
- Parser metadata identifies the path and may contain page count, parser version, and processing duration.
- `warnings` reports a fallback or partial issue; it should not be silently ignored in a real import.
- Parsing reads document structure; it does not decide which fields should overwrite the user's existing CV.

Try comparing the Markdown with the original CV. Tables, scanned pages, unusual layouts, and image-only PDFs can affect extraction quality.

In [ ]:
if markdown:
    print("Markdown is ready for extraction.")
else:
    markdown = input("Paste at least 10 characters of Markdown to continue (or press Enter to skip): ").strip()
    if len(markdown) < 10:
        markdown = None
        print("No Markdown selected; skip the extraction request cell.")
    else:
        print("Manual Markdown loaded; characters:", len(markdown))

## 4. Extract structured CV fields with AI

This calls `POST /api/cv/extract` with the Markdown. The server wraps document text as untrusted data in an extraction prompt, requests schema-constrained output, then validates it again against the CV schema. This is **inference**, not training. It may use Gemini or OpenRouter according to the server environment; the Ollama voice model does not automatically change this extractor.

The response includes `data`, `providerUsed`, processing time, and warnings. CV contents are sent to the configured AI service, so use test data if you do not want real CV text sent to a hosted provider.

In [ ]:
extracted = None
if markdown:
    response = requests.post(
        f"{APP_URL}/api/cv/extract/",
        json={"markdown": markdown, "filename": (parsed_cv or {}).get("filename", "notebook-demo")},
        timeout=150,
    )
    print("HTTP status:", response.status_code)
    if not response.ok:
        print("Error:", response.json().get("error", response.text[:500]))
    else:
        extraction_result = response.json()
        extracted = extraction_result.get("data")
        print("Provider:", extraction_result.get("providerUsed"))
        print("Processing ms:", extraction_result.get("processingTimeMs"))
        print("Warnings:", extraction_result.get("warnings", []))
        if extracted:
            print("Top-level fields:", list(extracted.keys()))
            print("Identity:", json.dumps(extracted.get("identity", {}), indent=2))
            print("Experience entries:", len(extracted.get("experience", [])))
            print("Skills groups:", len(extracted.get("skills", [])))
else:
    print("Skipped because Markdown is not set.")

## 5. What “AI training” means in this project

Uploading a CV does **not** fine-tune or train model weights. The actual lifecycle is:

1. AI extracts candidate facts from a document into the typed CV shape.
2. The owner reviews conflicts and edits the draft in the CV editor.
3. Publishing compiles the persona's public/unlocked knowledge and answer rules.
4. Normal-sized profiles are supplied as grounded context at answer time. Large profiles can use PostgreSQL/pgvector embeddings to retrieve relevant knowledge chunks. Embeddings are vectors used for search; they are not model-weight training.
5. At question time, the configured text model produces an answer from authorized profile context. The server validates routes/cards and applies privacy/leak checks before returning it.

Relevant code: `server/services/cv-extractor/cv-extractor.ts`, `server/brain/compile.ts`, `server/brain/embeddings.ts`, `server/brain/retrieval.ts`, `server/brain/answer.ts`, and `server/persona/publish.ts`. A fine-tuning pipeline would be a separate feature with training data, provider support, evaluation, and explicit data governance.

## 6. Test the live voice answer model

The browser voice interview uses browser speech recognition to turn microphone audio into text. Next.js `POST /api/voice/` sends the question and the published persona context server-to-server to the parser API. The parser API streams answer tokens from its configured `VOICE_MODELS`; local development can use Dockerized Ollama with `openai_compatible/llama3.2`. The browser then speaks the answer using native speech synthesis, or requests the server's Gemini TTS WAV fallback when native voices are missing.

The parser endpoint requires `X-API-Key`. Enter the **local development** key with `getpass` below; it is held in the kernel only, not written to the notebook. The example uses an abbreviated synthetic context unless you replace it with test CV data. The parser API is separate from the CV extraction Gemini/OpenRouter path.

Source files: `components/voice-interview.tsx`, `app/api/voice/route.ts`, `app/api/voice/speech/route.ts`, `parser-api/src/voice/router.py`, and `parser-api/src/voice/service.py`.

In [ ]:
parser_key = getpass("Local parser API key (input hidden; leave blank to skip): ")
voice_answer = ""

if parser_key:
    if extracted:
        # Use only a compact, reviewed subset for the demo context.
        context_data = {
            "identity": extracted.get("identity", {}),
            "title": extracted.get("title", ""),
            "profile": extracted.get("profile", ""),
            "experience": extracted.get("experience", [])[:3],
            "skills": extracted.get("skills", [])[:8],
            "projects": extracted.get("projects", [])[:3],
            "education": extracted.get("education", {}),
        }
        context = json.dumps(context_data, ensure_ascii=False)
    else:
        context = json.dumps({
            "identity": {"name": "Demo Candidate"},
            "profile": "A software engineer working on web applications and data tools.",
            "sections": [],
        })

    question = input("Question to ask the local voice model: ").strip() or "What kind of work do you do?"
    response = requests.post(
        f"{PARSER_API_URL}/api/v1/voice/stream",
        headers={"X-API-Key": parser_key},
        json={"question": question, "context": context, "history": []},
        stream=True,
        timeout=120,
    )
    print("HTTP status:", response.status_code)
    if not response.ok:
        print("Voice request failed:", response.text[:500])
    else:
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[len("data:"):].strip())
            if event.get("type") == "token":
                voice_answer += event.get("text", "")
                print(event.get("text", ""), end="", flush=True)
            elif event.get("type") == "error":
                print("\nVoice error:", event.get("message"))
        print("\nAnswer characters:", len(voice_answer))
    response.close()
else:
    print("Skipped. In the UI, test this from Ask AI by starting a live voice interview.")

## 7. Learn and test PayPal and Chapa billing safely

Billing is deliberately tested through the **signed-in web UI**, not by copying browser cookies into this notebook. The notebook must not initiate a charge or handle card/mobile-money details.

### CV Design Pass one-time checkout

- Browser calls `POST /api/billing/checkout/` with `{ "provider": "paypal" }` or `{ "provider": "chapa" }`.
- Server requires a customer session, validates configured credentials, creates a local pending order, then requests a provider checkout/approval URL.
- PayPal returns the user to the billing page; the UI calls `POST /api/billing/paypal/capture/`. The server captures/verifies the order and only then marks it paid.
- Chapa returns to the billing page; the UI calls `POST /api/billing/chapa/verify/`. The server verifies the transaction with Chapa and checks amount/currency before marking it paid. Chapa's callback route also verifies the transaction server-to-server.

### PayPal Portfolio Pro subscription

The subscription flow is separate from a CV Design Pass: the server creates a PayPal subscription against `PAYPAL_PORTFOLIO_PRO_PLAN_ID`, then listens for signature-verified PayPal webhooks (activate/update/cancel/suspend/expire) before updating local subscription state.

Relevant code: `app/api/billing/checkout/route.ts`, `app/api/billing/paypal/capture/route.ts`, `app/api/billing/chapa/verify/route.ts`, `app/api/billing/chapa/callback/route.ts`, `app/api/billing/paypal/subscription/route.ts`, `app/api/billing/paypal/webhook/route.ts`, and `server/payment-providers.ts`.

### Sandbox test checklist

1. Configure PayPal **sandbox** client ID/secret and a sandbox plan ID in the server environment. Keep `PAYPAL_ENVIRONMENT=sandbox`.
2. Configure Chapa test credentials in the server environment. Do not use live keys for this tutorial.
3. Set the correct public app URL and callback URLs. Provider callbacks/webhooks need a publicly reachable HTTPS endpoint during local testing; use an approved development tunnel and never expose credentials.
4. Sign in to the local app, open Account → Billing, and select a sandbox/test checkout.
5. Complete PayPal with a PayPal sandbox buyer, or Chapa with its test method. Return to Billing and observe pending → verified/paid status.
6. For subscriptions, configure the PayPal webhook URL and webhook ID, then verify activation/cancellation status changes.
7. Check server logs and the billing page; do not treat a redirect alone as proof of payment. The app verifies provider-side status and amount/currency.
8. Do not test a live payment or paste card numbers, wallet PINs, secrets, transaction tokens, or session cookies in this notebook.

The server reads `CHAPA_SECRET_KEY`, `PAYPAL_CLIENT_ID`, `PAYPAL_CLIENT_SECRET`, `PAYPAL_ENVIRONMENT`, `PAYPAL_PORTFOLIO_PRO_PLAN_ID`, and `PAYPAL_WEBHOOK_ID`. They must remain in server-side environment/secrets storage.

In [ ]:
import os

# Presence-only checks. Values are never shown or copied into notebook output.
for name in (
    "CHAPA_SECRET_KEY",
    "PAYPAL_CLIENT_ID",
    "PAYPAL_CLIENT_SECRET",
    "PAYPAL_PORTFOLIO_PRO_PLAN_ID",
    "PAYPAL_WEBHOOK_ID",
):
    print(f"{name}: {'configured in this kernel' if os.getenv(name) else 'not present in this kernel'}")
print("PAYPAL_ENVIRONMENT:", os.getenv("PAYPAL_ENVIRONMENT", "not present (server defaults to sandbox)"))
print("Note: the web app process has its own environment; this does not inspect its secrets.")

## 8. Troubleshooting and completion checklist

| Symptom | Check |
| --- | --- |
| App health request fails | Run `bun dev`; check port 3000. |
| CV parse returns Docling network error | `docker compose ps docling`; confirm `DOCLING_API_URL=http://localhost:5001` for the host-run web app. |
| Parser API unavailable | From `parser-api/`, check `docker compose ps api ollama` and `http://localhost:8000/api/v1/health`. |
| Voice answer fails | Check `VOICE_MODELS`, Ollama model list, API container URL `http://ollama:11434/v1`, and parser API key agreement. |
| CV extraction fails | Confirm the app process has a working `GEMINI_API_KEY` or `OPENROUTER_API_KEY`; inspect the server error logs without sharing key values. |
| Payment option is disabled | Check that server-side sandbox credentials are present and the app restarted after changes. |
| Payment remains pending | Check provider-side transaction, callback/webhook reachability, signature configuration, and server logs. A browser redirect alone is not proof of payment. |

### What you have tested

- [ ] A supported CV file was uploaded and parsed into Markdown.
- [ ] AI extraction returned schema-shaped fields and a provider name.
- [ ] You understood owner review/publish versus model training.
- [ ] Ollama streamed a grounded local voice answer, or you identified which prerequisite is missing.
- [ ] You traced the PayPal and Chapa sandbox routes without making a live payment.

Use fictional/test CV data where possible. The local Ollama model and Docling can run on your machine, but Gemini/OpenRouter extraction and Gemini TTS may send content to a hosted provider and incur usage. Remove only your own test data through the product's normal controls; do not delete shared Docker volumes to clean up.